---
title: "Get Started with Predictive Distribution"
description: "Learn how to extract predictive distributions with TabPFNRegressor"
cookbookTags:
  - regression
authors:
  - name: Prior Labs
    linkedin: https://www.linkedin.com/company/prior-labs
    twitter: https://twitter.com/prior_labs
---

*Going beyond point estimates to the full predictive distribution.*

A TabPFN regressor does not just output a single number per row. It predicts a full distribution over the target, which we can summarise as a mean, read off at chosen quantiles, or visualise directly. To generate point-estimates the `TabPFNRegressor` object takes the mean of the predicted distribution.

 This notebook captures that full output, inspects its structure, and uses it to draw calibrated uncertainty bands around the model's predictions. We run the regressor through the hosted API client (`tabpfn-client`); the same code works with the local `tabpfn` package by changing the import.

## Setup

*Installing the TabPFN client and the plotting dependencies.*

In [ ]:
!pip install tabpfn tabpfn-client matplotlib numpy scikit-learn

## Imports and Data

*Loading the diabetes dataset and the distribution plotting helper.*

The regressor comes from the hosted client (`tabpfn_client`). Alongside it we import `plot_regression_distribution` from the open-source `tabpfn` package, a helper that draws TabPFN's predicted distribution for a single sample.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from google.colab import userdata
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

from tabpfn_client import TabPFNRegressor
from tabpfn.visualisation import plot_regression_distribution


X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

## Authenticating with a Token

*Setting `TABPFN_TOKEN` so the client can reach the API.*

In [ ]:
from tabpfn_client import set_access_token
set_access_token(userdata.get('TABPFN_TOKEN'))

## Fit the Regressor

*A small ensemble keeps the full-distribution output light.*

Each estimator contributes its own predicted distribution, and the final output combines them. We use four here.

In [ ]:
reg = TabPFNRegressor(n_estimators=4)
reg.fit(X_train, y_train)

00:00 Fitting... \

00:00 Fitting... Done!


TabPFNRegressor(client_options=ClientOptions(timeout=900.0,
                                             headers={'sentry-trace': 'd7d5e153e9284a1aaa644ffd3cae054d'}),
                n_estimators=4)

## Capturing the Full Output

*Requesting `output_type="full"` returns a dictionary, not an array.*

Instead of one prediction per row, the full output exposes every summary TabPFN computes, keyed by name.

In [ ]:
preds = reg.predict(X_test, output_type="full")
preds.keys()  # preds are a dictionary

00:00 Predicting... \

00:04 Predicting... Done!


dict_keys(['mean', 'median', 'mode', 'quantiles', 'logits', 'borders', 'criterion'])

## Inspecting the Output

*The mean and the predicted quantiles, side by side.*

`mean` holds the point estimate per row. `quantiles` holds the predicted value at each default quantile, from the 10th to the 90th percentile.

In [ ]:
preds["mean"]

array([262.00595093, 247.46105957, 157.95018005, 112.13021088,
       168.45626831, 260.21682739, 101.82640839, 206.1414032 ,
       144.20509338, 239.69104004, 170.84356689, 186.34484863,
       102.5146637 ,  92.51793671, 267.41168213,  81.6782608 ,
       149.95092773,  75.79134369, 103.53370667, 226.96896362,
       193.64593506, 150.20507812, 163.20532227, 135.34480286,
       210.42773438, 168.1986084 , 108.92704773,  82.55252075,
       187.76705933, 157.13589478, 179.16030884,  83.51014709,
       136.45083618, 163.94586182, 146.91384888, 196.48248291,
       171.73265076, 183.18984985, 112.78544617, 208.20550537,
        90.22658539, 161.11459351, 141.42713928, 189.02024841,
       172.44181824,  82.47047424, 131.43049622, 123.00216675,
       109.7409668 , 240.20011902, 157.71350098,  76.81249237,
       135.29710388, 160.92269897, 243.48034668, 174.11050415,
       193.51113892, 108.92158508, 128.37950134, 182.95803833,
       227.03120422, 156.3691864 , 145.6862793 , 103.51

In [ ]:
# default quantiles are: [0.1, 0.2, ..., 0.8, 0.9]
# the shape of the quantiles array is quantiles x rows
preds["quantiles"].shape

(9, 89)

## Visualising a Single Prediction

*Plotting the full predicted distribution for one row.*

`plot_regression_distribution` shows the shape of the distribution TabPFN assigns to a single sample, making its confidence, or lack of it, visible at a glance.

In [ ]:
preds["logits"] = torch.as_tensor(np.nan_to_num(np.asarray(preds["logits"]), nan=-np.inf))
plot_regression_distribution(preds, sample_idx=0)

<Axes: title={'center': 'TabPFN predicted distribution'}, xlabel='Predicted target', ylabel='Probability density'>

![Predicted distribution for sample 0](../visuals/predictive_distribution/sample-prediction-0.png)


In [ ]:
plot_regression_distribution(preds, sample_idx=10)

<Axes: title={'center': 'TabPFN predicted distribution'}, xlabel='Predicted target', ylabel='Probability density'>

![Predicted distribution for sample 10](../visuals/predictive_distribution/sample-prediction-10.png)


## Uncertainty Across the Input Range

*Building an uncertainty band on a controlled synthetic dataset.*

To see how the input affects the uncertainty in the prediction, we generate data where the noise grows with the input.

### Generate the data

*A linear trend with noise that widens as X grows.*

In [ ]:
n_train = 100
n_test = 50
rng = np.random.default_rng(0)
X_train = rng.uniform(0.0, 10.0, size=n_train)
y_train = 2 * X_train + rng.uniform(size=n_train) * X_train
X_train = X_train.reshape(-1,1)
X_test = np.linspace(10.0, 20.0, n_test, dtype=np.float32).reshape(-1, 1)

### Generate predictions

*Mean predictions, plus the full distribution for the quantiles.*

In [ ]:
reg = TabPFNRegressor()
reg.fit(X_train, y_train)
preds = reg.predict(X_test) # mean point-estimates
full_preds = reg.predict(X_test, output_type="full")

00:00 Fitting... |

00:00 Fitting... Done!
00:00 Predicting... \

00:02 Predicting... Done!
00:00 Predicting... \

00:04 Predicting... Done!


### Plot the uncertainty band

*Shading between the 10th and 90th predicted quantiles.*

The band widens where the model is less certain, including the extrapolation region past the training data.

In [ ]:
# @title
q10 = full_preds["quantiles"][0]
q90 = full_preds["quantiles"][-1]

plt.scatter(X_train, y_train, label="Train Data")
plt.fill_between(X_test.flatten(), q10, q90, alpha=0.2, color="C1", label="Uncertainty Band")
plt.plot(X_test, preds, color="C1", label=" Mean Prediction")

plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.show()

![Uncertainty band plot](../visuals/predictive_distribution/uncertainty-band.png)
